In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")

BRONZE_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_bronze.order_items"
SILVER_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.order_items"
ORDERS_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.orders"
PRODUCTS_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.products"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.dq_quarantine"

QUARANTINE_SOURCE = "order_items"
AMOUNT_TOLERANCE_USD = 0.01

BRONZE_COLUMNS = [
    "order_id", "product_id", "unit_price_usd", "quantity", "line_total_usd",
    "_rescued_data", "source_filename", "ingestion_timestamp",
]
SILVER_COLUMNS = [
    "order_id", "line_number", "product_id", "unit_price_usd", "quantity",
    "line_total_usd", "source_filename", "ingestion_timestamp",
]

In [0]:
known_orders = spark.table(ORDERS_TABLE).select("order_id").withColumn("_order_known", F.lit(True))
known_products = spark.table(PRODUCTS_TABLE).select("product_id").withColumn("_product_known", F.lit(True))

expected_line_total = F.col("unit_price_usd") * F.col("quantity")

is_valid = (
    F.col("order_id").isNotNull()
    & F.col("product_id").isNotNull()
    & F.col("_rescued_data").isNull()
    & F.col("_order_known").isNotNull()
    & F.col("_product_known").isNotNull()
    & (F.col("quantity") > 0)
    & (F.col("unit_price_usd") > 0)
    & (F.abs(F.col("line_total_usd") - expected_line_total) <= AMOUNT_TOLERANCE_USD)
)

checked = (
    spark.table(BRONZE_TABLE)
    .select(*BRONZE_COLUMNS)
    .join(known_orders, "order_id", "left")
    .join(known_products, "product_id", "left")
    .withColumn("_is_valid", F.coalesce(is_valid, F.lit(False)))
)

line_order = Window.partitionBy("order_id").orderBy(
    "product_id", "unit_price_usd", "quantity", "line_total_usd"
)

valid_items = (
    checked.filter("_is_valid")
    .withColumn("line_number", F.row_number().over(line_order))
    .select(*SILVER_COLUMNS)
    .withColumn("processed_at", F.current_timestamp())
)
invalid_items = checked.filter("NOT _is_valid")

In [0]:
(
    DeltaTable.forName(spark, SILVER_TABLE).alias("t")
    .merge(valid_items.alias("s"), "t.order_id = s.order_id AND t.line_number = s.line_number")
    .whenMatchedUpdateAll(condition="s.ingestion_timestamp > t.ingestion_timestamp")
    .whenNotMatchedInsertAll()
    .execute()
)

quarantine_records = invalid_items.select(
    F.lit(QUARANTINE_SOURCE).alias("source_table"),
    F.to_json(F.struct(*BRONZE_COLUMNS)).alias("record"),
    F.current_timestamp().alias("quarantined_at"),
)

(
    quarantine_records.write
    .mode("overwrite")
    .option("replaceWhere", f"source_table = '{QUARANTINE_SOURCE}'")
    .saveAsTable(QUARANTINE_TABLE)
)